# Four more leagues: the same gap, and a better XGBoost

Companion notebook to the [four-league report](https://github.com/JHvL88/matchday/blob/main/analysis/backtest/leagues/README.md) of [Jan's Matchday](https://jhvl88.github.io/matchday/).
It runs the Bundesliga models, unchanged, on England, Spain, Italy and France as well, and trains XGBoost on all five leagues at once.
The questions and decision rules were [fixed beforehand](https://github.com/JHvL88/matchday/blob/main/analysis/backtest/leagues/PREREGISTRATION.md).

1. **Source data** for five leagues, as stored on the `source-data` branch, and one match table per league.
2. **Pooled XGBoost**, refitted here (about five minutes; on a GPU if the runtime has one).
3. **Grades**: the gap to the closing market per league, pooled against own XGBoost, the pre-registered rules.

**Runtime:** about ten minutes as it is. The switches below refit more:
`RUN_DC_XGB = True` adds about 20 minutes; `RUN_BAYES = True` refits the Bayesian model 367 times (Runtime › Change runtime type › GPU; expect several hours).
With a switch off, that model's forecasts are loaded from the published run.

The model code lives in [`analysis/pipeline/`](https://github.com/JHvL88/matchday/tree/main/analysis/pipeline), the same files the posts use.

In [ ]:
RUN_POOLED = True      # refit XGBoost on all five leagues (about 5 minutes)
RUN_DC_XGB = False     # refit Dixon-Coles and each league's own XGBoost (about 20 minutes)
RUN_BAYES = False      # refit the Bayesian model for every league and month (hours, GPU)
FIRST, LAST = 2019, 2026

import json, os, shutil, subprocess, sys
REPO = "https://github.com/JHvL88/matchday"
BRANCH = os.environ.get("MATCHDAY_BRANCH", "main")
LEAGUES = ["GER", "ENG", "ESP", "ITA", "FRA"]

def sh(*cmd, quiet=False):
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if (not quiet or r.returncode) and (r.stdout + r.stderr).strip():
        print((r.stdout + r.stderr).strip())
    if r.returncode:
        raise RuntimeError(f"failed: {' '.join(str(c) for c in cmd)}")
    return r.stdout

RESULTS_COMMIT = "7124cc7aa63d85ec82207560399b6f93c4ac9c6a"   # backtest-results commit behind the report

for folder, ref in [("matchday", BRANCH), ("source", "source-data")]:
    if not os.path.exists(folder):
        sh("git", "clone", "-q", "--depth", "1", "-b", ref, REPO, folder)
if not os.path.exists("published"):      # the predictions exactly as the report graded them
    sh("git", "init", "-q", "published")
    sh("git", "-C", "published", "fetch", "-q", "--depth", "1", REPO, RESULTS_COMMIT)
    sh("git", "-C", "published", "checkout", "-q", "FETCH_HEAD")
PIPE = "matchday/analysis/pipeline"
sys.path.insert(0, PIPE)
if RUN_BAYES or RUN_DC_XGB:
    sh(sys.executable, "-m", "pip", "install", "-q", "numpyro", "xgboost", quiet=True)   # Colab keeps its own GPU build of jax
print("code:", sh("git", "-C", "matchday", "log", "-1", "--format=%h %s", quiet=True).strip())
print("source data:", sh("git", "-C", "source", "log", "-1", "--format=%h %s", quiet=True).strip())
print("published run:", json.load(open("published/leagues/run.json"))["run_id"])

### One match table per league

football-data.co.uk's files for the top two tiers of each league. `div` is the tier in every table (D1 top flight, D2 second tier), so the models run on any league unchanged.

In [ ]:
import pandas as pd
os.makedirs("tables", exist_ok=True)
for league in LEAGUES:
    sh(sys.executable, f"{PIPE}/build_matches.py", "source", f"tables/{league}.csv", "--league", league, quiet=True)
rows = []
for league in LEAGUES:
    m = pd.read_csv(f"tables/{league}.csv", parse_dates=["date"])
    test = m[(m["div"] == "D1") & (m.season >= FIRST)]
    rows.append(dict(league=league, matches=len(m), test_matches=len(test), with_closing_odds=int(test.avgc_h.notna().sum()),
                     last_match=f"{m.date.max():%Y-%m-%d}"))
pd.DataFrame(rows)

### The models

Forecasts start as a copy of the published run. Each switch replaces one part with a refit on this machine.

In [ ]:
if os.path.exists("results"):
    shutil.rmtree("results")
shutil.copytree("published/leagues", "results")
seasons = [str(FIRST), str(LAST)]
if RUN_DC_XGB:
    for league in LEAGUES:
        sh(sys.executable, f"{PIPE}/backtest.py", "dc", f"tables/{league}.csv", f"results/{league}/oos_dc.csv", "--seasons", *seasons)
        sh(sys.executable, f"{PIPE}/backtest.py", "xgb", f"tables/{league}.csv", f"results/{league}/oos_xgb.csv", "--seasons", *seasons)
if RUN_BAYES:
    for league in LEAGUES:
        for season in range(FIRST, LAST + 1):
            sh(sys.executable, f"{PIPE}/backtest.py", "bayes", f"tables/{league}.csv", f"results/{league}/oos_bay_{season}.csv",
               "--seasons", season, season)
if RUN_POOLED:
    tables = [x for league in LEAGUES for x in ("--table", league, f"tables/{league}.csv")]
    print(sh(sys.executable, f"{PIPE}/pooled_xgb.py", "results/pooled/oos_xgb_pooled.csv", "--seasons", *seasons, *tables,
             quiet=True).strip().splitlines()[-1])
    mine = pd.read_csv("results/pooled/oos_xgb_pooled.csv").set_index(["league", "key"])
    theirs = pd.read_csv("published/leagues/pooled/oos_xgb_pooled.csv").set_index(["league", "key"]).loc[mine.index]
    print(f"Largest difference from the published pooled forecasts: {(mine - theirs).abs().max().max():.6f} goals "
          "(0 on a CPU; a GPU can shift the fourth decimal)")

### Grades

RPS (ranked probability score) measures how far the home/draw/away probabilities were from what happened; lower is better.
The benchmark is the market-average closing price with the margin removed. Differences come with a paired-bootstrap 95% interval.

Q2's rule is graded on the Bundesliga's xG models. Their forecasts come from the published runs: the xG itself is in a private repository, because Understat publishes no licence.

In [ ]:
print(sh(sys.executable, f"{PIPE}/evaluate_leagues.py", "tables", "results", "results.json",
         "--xg-table", "tables/GER.csv", "--xg-results", "published/xg", quiet=True))
res = json.load(open("results.json"))
pub = json.load(open("matchday/analysis/backtest/leagues/results.json"))
same = all(abs(res[k]["diff"] - pub[k]["diff"]) < 1e-9 for k in ["q1", "q2"])
print("Same answers as the published report." if same else "Differs from the published report (expected after a refit or new matches):",
      "" if same else {k: (round(res[k]["diff"], 5), round(pub[k]["diff"], 5)) for k in ["q1", "q2"]})

In [ ]:
from IPython.display import SVG, display
sh(sys.executable, f"{PIPE}/make_league_figures.py", "results.json", "figs", quiet=True)
for name in ["gap_by_league", "pooled_by_league", "xgboost_place", "gap_by_season"]:
    display(SVG(f"figs/{name}.svg"))